# TP 3 - MapReduce avancé : chiffre d'affaires et qualité des données

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syhard87/tp_hadloop/blob/claude/tp-hadoop-avances-partie-a-mwgfq7/notebooks/TP3_MapReduce_Qualite.ipynb)

Source : `sources/TP_Hadoop_avances_avec_TP_ecosysteme.pdf`, pages 14 et 15. Guide de l'oral : `docs/TP3_Guide_oral.md`.

**Mission** : développer un traitement **Hadoop Streaming** en Python qui calcule le **chiffre d'affaires par pays** et
détecte les lignes invalides grâce à des **compteurs métier**.

```
ventes.csv ──put──▶ HDFS /tp_avance/ca/input ──▶ MAP (mapper.py) ──▶ SHUFFLE/SORT ──▶ REDUCE (reducer.py) ──▶ HDFS /tp_avance/ca/output
  (local)                                         contrôle qualité      regroupe par pays     somme par pays          part-00000 + _SUCCESS
                                                  émet  pays<TAB>montant                                                 + compteurs Qualite
```

**Mode d'emploi** : se connecter (coche verte en haut à droite), puis exécuter les cellules **dans l'ordre**.
Si la session redémarre, reprendre depuis la cellule 1.

> **Message `Connection refused` sur `localhost:9000`** : HDFS ne tourne pas. Relancer la **cellule 2**, qui redémarre
> uniquement les démons arrêtés sans effacer les données, et attendre `✅ Hadoop prêt`.

## 1. Récupérer le dépôt (code, données, script d'installation)

In [ ]:
!rm -rf /content/sample_data /content/tp_hadloop
!git clone -q --depth 1 -b claude/tp-hadoop-avances-partie-a-mwgfq7 https://github.com/Syhard87/tp_hadloop.git /content/tp_hadloop
!ls -l /content/tp_hadloop/tp3 /content/tp_hadloop/data

## 2. Installer et démarrer Hadoop (HDFS + YARN)
Il faut **YARN** pour exécuter un job MapReduce : le ResourceManager distribue les tâches, et les NodeManagers les exécutent
dans des conteneurs. **Durée** : 2 à 4 minutes à la première exécution.

In [ ]:
import os
HADOOP_HOME = "/content/hadoop-3.5.0"
os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
if f"{HADOOP_HOME}/bin" not in os.environ["PATH"]:
    os.environ["PATH"] = f"{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:" + os.environ["PATH"]
os.makedirs("/content/preuves_tp3", exist_ok=True)

!bash /content/tp_hadloop/scripts/setup_hadoop_colab.sh

# Arrête « Tout exécuter » ici si Hadoop n'a pas démarré (sinon les cellules suivantes échoueraient en silence)
import subprocess
if "OFF" not in subprocess.run(["hdfs", "dfsadmin", "-safemode", "get"], capture_output=True, text=True).stdout:
    raise RuntimeError("Hadoop n'est pas prêt : lire le message ❌ ci-dessus, puis relancer cette cellule")

## 3. Préparer le dossier de travail local
`/content/tp3` contient le Mapper, le Reducer et `ventes.csv` (les données de l'énoncé, page 14, séparateur `;`).
`chmod +x` rend les scripts exécutables : c'est nécessaire pour `./mapper.py`, et la 1re ligne `#!/usr/bin/env python3` indique l'interpréteur.

In [ ]:
%%bash
mkdir -p /content/tp3 && cd /content/tp3
cp /content/tp_hadloop/tp3/mapper.py /content/tp_hadloop/tp3/reducer.py .
cp /content/tp_hadloop/data/ventes_tp3.csv ventes.csv
chmod +x mapper.py reducer.py
ls -l
echo; cat ventes.csv

## Mission A - Mapper (questions 15 à 18)

| Question | Où dans `mapper.py` |
|---|---|
| **15.** Lire chaque ligne depuis l'entrée standard | `for brute in sys.stdin:` (Hadoop envoie une ligne du split par ligne de stdin) |
| **16.** Contrôler le format et les règles de qualité | 5 champs, pays non vide, quantité entière > 0, prix numérique > 0, en-tête ignoré |
| **17.** Émettre pays et montant pour les lignes valides | `print(f"{pays}\t{montant}")` : la **clé** et la **valeur** sont séparées par une **tabulation** |
| **18.** Un compteur différent par anomalie | `reporter:counter:Qualite,<Nom>,1` écrit sur **stderr** : `Pays_manquant`, `Quantite_invalide`, `Valeur_negative` (+ `Prix_invalide`, `Format_invalide`) |

Choix importants :
- Chaque ligne rejetée incrémente **un seul** compteur, celui de la 1re règle violée. On a donc l'égalité :
  lignes lues = en-tête + valides + rejetées.
- Les montants sont calculés en **`Decimal`** (calcul exact, alors qu'avec un `float`, 0,1 + 0,2 ≠ 0,3) et émis **sans arrondi** :
  l'arrondi est fait une seule fois, à la fin, par le Reducer.
- Le séparateur `;` ou `,` est détecté automatiquement. Les `\r` (fichiers Windows) et le BOM sont retirés.

In [ ]:
!cat -n /content/tp3/mapper.py

## Mission B - Reducer
- **Additionner les montants par pays** : Hadoop garantit que l'entrée du Reducer est **triée par clé**. Toutes les lignes
  d'un pays sont donc **consécutives**. On cumule tant que la clé ne change pas, et on émet le total dès qu'elle change,
  puis une dernière fois en fin de flux.
- **Arrondi à deux décimales** : `quantize(Decimal("0.01"), ROUND_HALF_UP)`, l'arrondi commercial (0,005 → 0,01).
- **Pas de liste de pays codée en dur** : les groupes sont découverts dynamiquement par changement de clé. Un nouveau pays
  dans les données apparaît automatiquement dans le résultat.
- La mémoire utilisée est constante, quel que soit le volume : on ne garde qu'un pays et un total à la fois.

In [ ]:
!cat -n /content/tp3/reducer.py

## Mission C - Test local (sans Hadoop)
On simule le job avec des **tubes Unix** : `cat` joue la lecture HDFS, `./mapper.py` la phase Map, `sort` le Shuffle/Sort
et `./reducer.py` la phase Reduce. Ce test est rapide et permet de déboguer avant de lancer un vrai job.

In [ ]:
%%bash
cd /content/tp3
# 1) Mapper seul : stdout (paires clé/valeur) et stderr (compteurs + rejets) dans deux fichiers séparés
cat ventes.csv | ./mapper.py 1> sortie_map.txt 2> erreurs_map.txt

echo "===== sortie_map.txt (stdout du Mapper : $(wc -l < sortie_map.txt) paires, dans l'ordre du fichier)"
cat sortie_map.txt
echo
echo "===== erreurs_map.txt (stderr du Mapper : compteurs et lignes rejetées)"
cat erreurs_map.txt

In [ ]:
%%bash
cd /content/tp3
echo "===== Après sort (simulation du Shuffle/Sort) : les clés identiques sont regroupées"
cat ventes.csv | ./mapper.py 2>/dev/null | sort
echo
echo "===== Résultat final : cat | mapper | sort | reducer"
cat ventes.csv | ./mapper.py | sort | ./reducer.py 2>/dev/null | tee resultat_local.txt
cp sortie_map.txt erreurs_map.txt resultat_local.txt /content/preuves_tp3/

**Lecture de la sortie** : dans la 2e commande, le stderr du Mapper n'est pas redirigé. Les lignes `reporter:counter…` et
`REJET…` s'affichent donc à l'écran, mais **ne passent pas** dans le tube vers `sort`, qui ne reçoit que stdout.
`resultat_local.txt` ne contient que les 3 lignes du Reducer.

**Mapper ou résultat final** : le Mapper produit **6 paires**, une par ligne valide, **dans l'ordre du fichier**, avec
des pays répétés. Le résultat final contient **3 lignes**, une par pays, **triées** et **agrégées**. Le Mapper transforme
et filtre ; le Reducer agrège.

**Pourquoi `sort` simule le Shuffle/Sort** : entre Map et Reduce, Hadoop **partitionne** les paires par clé, les **trie
par clé**, les **transfère** à chaque Reducer et les **fusionne**. Le Reducer reçoit donc toutes les valeurs d'une même clé
**à la suite**. `sort` produit le même effet : la clé est en début de ligne, donc les lignes d'un même pays deviennent
**adjacentes**. C'est l'hypothèse sur laquelle repose `reducer.py`. Sans `sort`, le Reducer produirait 6 lignes (FR 3 fois, BE 2 fois),
avec des totaux faux.

### Vérification manuelle des totaux

| Ligne | Pays | Quantité × Prix | Montant | Statut |
|---|---|---|---|---|
| 2 | FR | 2 × 10,00 | 20,00 | ✅ |
| 3 | BE | 1 × 15,00 | 15,00 | ✅ |
| 4 | FR | 3 × 12,00 | 36,00 | ✅ |
| 5 | BE | 4 × 10,00 | 40,00 | ✅ |
| 6 | ES | 2 × 15,00 | 30,00 | ✅ |
| 7 | FR | 1 × 10,00 | 10,00 | ✅ |
| 8 | DE | `abc` × 20,00 | - | ❌ `Quantite_invalide` |
| 9 | *(vide)* | 2 × 9,50 | - | ❌ `Pays_manquant` |
| 10 | FR | -1 × 30,00 | - | ❌ `Valeur_negative` |

- **FR** = 20 + 36 + 10 = **66,00**
- **BE** = 15 + 40 = **55,00**
- **ES** = **30,00**
- **DE** n'apparaît pas : sa seule ligne est invalide.

La cellule suivante refait le calcul **indépendamment** (pandas) et le compare au résultat du Reducer.

In [ ]:
import pandas as pd
df = pd.read_csv("/content/tp3/ventes.csv", sep=";", dtype=str, keep_default_na=False)
q = pd.to_numeric(df["quantite"], errors="coerce")
p = pd.to_numeric(df["prix_unitaire"], errors="coerce")
valides = df[(df["pays"] != "") & (q > 0) & (q == q.round()) & (p > 0)].copy()
valides["montant"] = q[valides.index] * p[valides.index]
attendu = valides.groupby("pays")["montant"].sum().round(2)
print("Calcul indépendant (pandas) :"); print(attendu.to_string())

obtenu = pd.read_csv("/content/tp3/resultat_local.txt", sep="\t", header=None, names=["pays", "ca"]).set_index("pays")["ca"]
print("\nIdentique au Reducer :", attendu.equals(obtenu))

## Insertion des données dans HDFS (avant la Mission D)
Un job MapReduce lit ses entrées **dans HDFS**, pas sur le disque local. On crée le dossier d'entrée et on y dépose le fichier.

**Ce qui se passe lors d'un `put`** :
1. Le client demande au **NameNode** de créer le fichier. Le NameNode vérifie les droits et que le chemin est libre.
2. Le NameNode attribue un **bloc** et choisit les **DataNodes** de destination (ici un seul, réplication 1).
3. Le client envoie les données **directement au DataNode**, en paquets. Avec une réplication de 3, ce serait un
   *pipeline* DN1 → DN2 → DN3 avec des accusés de réception.
4. Une fois l'écriture terminée, le NameNode **enregistre** le fichier dans ses métadonnées (*edit log*). Le fichier est alors visible.

In [ ]:
%%bash
set -e   # la cellule s'arrête en erreur si une commande HDFS échoue
cd /content/tp3
hdfs dfs -mkdir -p /tp_avance/ca/input
hdfs dfs -put -f ventes.csv /tp_avance/ca/input/
echo "=== Présence (ls)";            hdfs dfs -ls -R /tp_avance
echo "=== Lecture (cat)";            hdfs dfs -cat /tp_avance/ca/input/ventes.csv
echo "=== Métadonnées (stat)";       hdfs dfs -stat "nom=%n taille=%b replication=%r bloc=%o" /tp_avance/ca/input/ventes.csv
echo "=== Blocs (fsck)";             hdfs fsck /tp_avance/ca/input/ventes.csv -files -blocks 2>/dev/null | grep -E "bytes|len="

## Mission D - Exécution Hadoop

| Option | Rôle |
|---|---|
| `STREAMING_JAR=$(find …)` | Trouve le jar **Hadoop Streaming**, qui permet d'écrire Mapper et Reducer dans n'importe quel langage lisant stdin et écrivant stdout |
| `hdfs dfs -rm -r -f …/output` | Supprime l'ancien résultat : **Hadoop refuse d'écrire dans un dossier output existant** |
| `-D mapreduce.job.name=…` | Nom du job, visible dans YARN (`yarn application -list`) |
| `-D mapreduce.job.reduces=1` | Un seul Reducer, donc **un seul fichier** de sortie `part-00000`, trié par pays |
| `-files mapper.py,reducer.py` | Copie les scripts dans le **cache distribué** : chaque conteneur en reçoit une copie locale |
| `-mapper "python3 mapper.py"` | Commande lancée par chaque tâche Map |
| `-reducer "python3 reducer.py"` | Commande lancée par chaque tâche Reduce |
| `-input` / `-output` | Chemins **HDFS** d'entrée et de sortie |
| `2>&1 \| tee execution.log` | Affiche le journal et l'enregistre (compteurs pour la Mission E) |

In [ ]:
%%bash
set -eo pipefail   # la cellule s'arrête en erreur si le job échoue (pipefail : même à travers « | tee »)
cd /content/tp3
STREAMING_JAR=$(find "$HADOOP_HOME/share/hadoop/tools/lib" -name 'hadoop-streaming-*.jar' | head -1)
echo "STREAMING_JAR=$STREAMING_JAR"
hdfs dfs -rm -r -f /tp_avance/ca/output
hadoop jar "$STREAMING_JAR" -D mapreduce.job.name=TP_Avance_CA_Qualite -D mapreduce.job.reduces=1 \
  -files mapper.py,reducer.py -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
  -input /tp_avance/ca/input -output /tp_avance/ca/output \
  2>&1 | tee execution.log
cp execution.log /content/preuves_tp3/

In [ ]:
%%bash
echo "=== Contenu du dossier de sortie"
hdfs dfs -ls /tp_avance/ca/output
echo
echo "=== Résultat (lu dans HDFS)"
hdfs dfs -cat /tp_avance/ca/output/part-00000 | tee /content/preuves_tp3/resultat_hadoop.txt
echo
echo "=== Comparaison avec le test local"
diff /content/tp3/resultat_local.txt /content/preuves_tp3/resultat_hadoop.txt && echo "IDENTIQUE au test local"

`_SUCCESS` est un fichier vide écrit **à la fin** d'un job réussi. Comme `_READY` au TP 2, il sert de marqueur pour les
traitements suivants. `part-00000` est la sortie du Reducer n° 0.

## Mission E - Analyse (questions 19 à 22)
### 19-20. Compteurs du framework et compteurs métier

In [ ]:
import re, pandas as pd
log = open("/content/tp3/execution.log", encoding="utf-8", errors="replace").read()
def compteur(nom):
    m = re.search(rf"^\s+{re.escape(nom)}=(\d+)", log, re.M)
    return int(m.group(1)) if m else None

framework = ["Map input records", "Map output records", "Reduce input groups", "Reduce input records",
             "Reduce output records", "Launched map tasks", "Launched reduce tasks", "Spilled Records"]
metier = ["En_tete", "Lignes_valides", "Pays_manquant", "Quantite_invalide", "Valeur_negative",
          "Prix_invalide", "Format_invalide"]
t1 = pd.DataFrame({"Compteur": framework, "Valeur": [compteur(c) for c in framework]})
t2 = pd.DataFrame({"Compteur (groupe Qualite)": metier, "Valeur": [compteur(c) or 0 for c in metier]})
t1.to_csv("/content/preuves_tp3/compteurs_framework.csv", index=False)
t2.to_csv("/content/preuves_tp3/compteurs_metier.csv", index=False)
display(t1); display(t2)

rejetees = sum(compteur(c) or 0 for c in ["Pays_manquant", "Quantite_invalide", "Valeur_negative", "Prix_invalide", "Format_invalide"])
print(f"Contrôle : lues {compteur('Map input records')} = en-tête {compteur('En_tete')} + valides {compteur('Lignes_valides')} + rejetées {rejetees}")

### 21. Lignes lues, valides, rejetées et groupes Reduce

| Notion | Compteur | Valeur | Explication |
|---|---|---|---|
| **Lignes lues** | `Map input records` | **10** | Toutes les lignes du fichier, **en-tête compris** (9 ventes + 1 en-tête). C'est le framework qui compte. |
| **Lignes valides** | `Map output records` = `Lignes_valides` | **6** | Lignes qui passent toutes les règles. Chacune produit **une** paire `pays<TAB>montant`. |
| **Lignes rejetées** | `Pays_manquant` + `Quantite_invalide` + `Valeur_negative` | **3** | 1 + 1 + 1, sans sortie. L'en-tête est **ignoré**, pas rejeté. |
| **Groupes Reduce** | `Reduce input groups` = `Reduce output records` | **3** | Nombre de **clés distinctes** (BE, ES, FR). Chaque groupe produit une ligne de résultat. |

**Relation** : 10 lues = 1 en-tête + 6 valides + 3 rejetées. Les 6 paires sont regroupées en 3 groupes par le Shuffle/Sort.
Le nombre de groupes dépend du **nombre de pays distincts**, pas du nombre de lignes.

In [ ]:
%%bash
echo "=== L'application dans YARN"
yarn application -list -appStates ALL 2>/dev/null | grep -E "Application-Id|TP_Avance" | cut -c1-130
APP=$(grep -o 'application_[0-9]*_[0-9]*' /content/tp3/execution.log | head -1)
echo; echo "=== Lignes rejetées, lues dans les journaux des conteneurs Map (yarn logs -applicationId $APP)"
sleep 5
yarn logs -applicationId $APP 2>/dev/null | grep "REJET" | tee /content/preuves_tp3/rejets_yarn_logs.txt

### 22. Provoquer l'erreur « dossier output existant », puis la corriger
On relance **exactement le même job sans supprimer** `/tp_avance/ca/output`.

In [ ]:
%%bash
cd /content/tp3
STREAMING_JAR=$(find "$HADOOP_HOME/share/hadoop/tools/lib" -name 'hadoop-streaming-*.jar' | head -1)
hadoop jar "$STREAMING_JAR" -D mapreduce.job.name=TP_Avance_CA_Qualite -D mapreduce.job.reduces=1 \
  -files mapper.py,reducer.py -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
  -input /tp_avance/ca/input -output /tp_avance/ca/output \
  > erreur_output.log 2>&1
echo "Code retour : $?"
grep -E "ERROR|Exception" erreur_output.log | cut -c1-200 | tee /content/preuves_tp3/erreur_output_existant.txt

**Diagnostic** : `Output directory hdfs://localhost:9000/tp_avance/ca/output already exists`, avec un code retour non nul.
Hadoop vérifie ce point **avant même de soumettre** le job à YARN (`FileOutputFormat.checkOutputSpecs`) : aucune application
n'est créée. C'est une **protection** contre l'écrasement accidentel d'un résultat, qui peut avoir coûté des heures de calcul.

**Deux corrections possibles** :
1. Supprimer l'ancien résultat, comme le fait le script de la Mission D : `hdfs dfs -rm -r -f /tp_avance/ca/output`.
2. Garder l'ancien résultat et écrire ailleurs : `-output /tp_avance/ca/output_v2`, ou un dossier daté.

In [ ]:
%%bash
cd /content/tp3
STREAMING_JAR=$(find "$HADOOP_HOME/share/hadoop/tools/lib" -name 'hadoop-streaming-*.jar' | head -1)
hdfs dfs -rm -r -f /tp_avance/ca/output           # correction
hadoop jar "$STREAMING_JAR" -D mapreduce.job.name=TP_Avance_CA_Qualite -D mapreduce.job.reduces=1 \
  -files mapper.py,reducer.py -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
  -input /tp_avance/ca/input -output /tp_avance/ca/output \
  > correction.log 2>&1
echo "Code retour : $?"
grep -E "completed successfully" correction.log
hdfs dfs -cat /tp_avance/ca/output/part-00000

## Cycle de vie des données dans HDFS : lecture, modification, suppression
**HDFS est « write-once, read-many »** : on ne peut **pas modifier un fichier au milieu**. Il n'existe que trois façons de
changer des données :
- **ajouter à la fin** (`-appendToFile`) ;
- **remplacer** le fichier entier (`-put -f`) ;
- **tronquer** (`-truncate`).

Un résultat MapReduce n'est **jamais mis à jour** : on le **recalcule**.

In [ ]:
%%bash
cd /content/tp3
STREAMING_JAR=$(find "$HADOOP_HOME/share/hadoop/tools/lib" -name 'hadoop-streaming-*.jar' | head -1)
echo "=== LECTURE : tail (fin du fichier) et get (copie HDFS -> local)"
hdfs dfs -tail /tp_avance/ca/input/ventes.csv | tail -n 2
hdfs dfs -get -f /tp_avance/ca/output/part-00000 resultat_recupere.txt && cat resultat_recupere.txt

echo; echo "=== MODIFICATION : ajout d'une vente en fin de fichier (appendToFile)"
echo "2026-10-06;IT;P07;5;8.00" > nouvelle_vente.csv
hdfs dfs -stat "taille avant : %b octets" /tp_avance/ca/input/ventes.csv
hdfs dfs -appendToFile nouvelle_vente.csv /tp_avance/ca/input/ventes.csv
hdfs dfs -stat "taille apres : %b octets" /tp_avance/ca/input/ventes.csv
hdfs dfs -tail /tp_avance/ca/input/ventes.csv | tail -n 2

echo; echo "=== Le résultat n'est pas mis à jour automatiquement : on relance le job"
hdfs dfs -rm -r -f /tp_avance/ca/output > /dev/null
hadoop jar "$STREAMING_JAR" -D mapreduce.job.name=TP_Avance_CA_Qualite -D mapreduce.job.reduces=1 \
  -files mapper.py,reducer.py -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
  -input /tp_avance/ca/input -output /tp_avance/ca/output \
  2>&1 | grep -E "Map input records|Map output records|Reduce output records"
hdfs dfs -cat /tp_avance/ca/output/part-00000

echo; echo "=== REMPLACEMENT : retour au fichier d'origine (put -f écrase)"
hdfs dfs -put -f ventes.csv /tp_avance/ca/input/ventes.csv
hdfs dfs -stat "taille : %b octets" /tp_avance/ca/input/ventes.csv

echo; echo "=== COPIE, RENOMMAGE, SUPPRESSION"
hdfs dfs -cp /tp_avance/ca/output/part-00000 /tp_avance/ca/resultat_copie.txt
hdfs dfs -mv /tp_avance/ca/resultat_copie.txt /tp_avance/ca/resultat_v1.txt
hdfs dfs -ls /tp_avance/ca
hdfs dfs -rm /tp_avance/ca/resultat_v1.txt
echo "Corbeille (fs.trash.interval, en minutes ; 0 = désactivée, donc suppression définitive) : $(hdfs getconf -confKey fs.trash.interval)"

**Ce qui se passe lors d'une lecture** : le client demande au NameNode **où sont les blocs**. Le NameNode ne renvoie que
des **adresses**, puis le client lit les données **directement sur les DataNodes**. Les données ne transitent jamais par le
NameNode.

**Ce qui se passe lors d'une suppression** : le NameNode retire le fichier de ses **métadonnées** (l'opération est immédiate).
Il ordonne ensuite aux DataNodes, via la réponse à leur **heartbeat**, d'effacer les blocs physiques, de façon asynchrone.
Si la corbeille était activée (`fs.trash.interval > 0`), le fichier serait d'abord déplacé dans `/user/<user>/.Trash`.

## Bonus - Le même code sur les ventes de l'enseignant (`data/ventes.csv`, séparateur `,`)
Même Mapper et même Reducer, sans aucune modification : la liste des pays n'est pas codée en dur et le séparateur est détecté.

In [ ]:
%%bash
cd /content/tp3
cp /content/tp_hadloop/data/ventes.csv ventes_enseignant.csv
STREAMING_JAR=$(find "$HADOOP_HOME/share/hadoop/tools/lib" -name 'hadoop-streaming-*.jar' | head -1)
hdfs dfs -mkdir -p /tp_avance/ca_enseignant/input
hdfs dfs -put -f ventes_enseignant.csv /tp_avance/ca_enseignant/input/
hdfs dfs -rm -r -f /tp_avance/ca_enseignant/output > /dev/null
hadoop jar "$STREAMING_JAR" -D mapreduce.job.name=TP_Avance_CA_Enseignant -D mapreduce.job.reduces=1 \
  -files mapper.py,reducer.py -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
  -input /tp_avance/ca_enseignant/input -output /tp_avance/ca_enseignant/output \
  2>&1 | grep -E "Map input records|Map output records|Reduce input groups|Reduce shuffle bytes|En_tete|Lignes_valides"
hdfs dfs -cat /tp_avance/ca_enseignant/output/part-00000 | tee /content/preuves_tp3/resultat_enseignant.txt
echo; echo "Lignes en double non détectées par les règles du TP 3 :"; sort ventes_enseignant.csv | uniq -d

### Bonus - Combiner et plusieurs Reducers
- **Combiner** (`-combiner`) : un « mini-Reducer » exécuté **côté Map**, avant le Shuffle. La somme étant **associative et
  commutative**, le Reducer peut servir de Combiner. Il réduit le volume transféré sur le réseau.
- **`mapreduce.job.reduces=2`** : le **Partitioner** (par défaut, le hachage de la clé modulo le nombre de Reducers) répartit
  les pays entre 2 Reducers, d'où 2 fichiers `part-0000x`. Un même pays va **toujours** au même Reducer.

In [ ]:
%%bash
cd /content/tp3
STREAMING_JAR=$(find "$HADOOP_HOME/share/hadoop/tools/lib" -name 'hadoop-streaming-*.jar' | head -1)
hdfs dfs -rm -r -f /tp_avance/ca_enseignant/output_combiner /tp_avance/ca_enseignant/output_2r > /dev/null
echo "=== Avec combiner"
hadoop jar "$STREAMING_JAR" -D mapreduce.job.reduces=1 -files mapper.py,reducer.py \
  -mapper "python3 mapper.py" -combiner "python3 reducer.py" -reducer "python3 reducer.py" \
  -input /tp_avance/ca_enseignant/input -output /tp_avance/ca_enseignant/output_combiner \
  2>&1 | grep -E "Map output records|Combine input records|Combine output records|Reduce shuffle bytes|Reduce input records"
diff <(hdfs dfs -cat /tp_avance/ca_enseignant/output/part-00000) <(hdfs dfs -cat /tp_avance/ca_enseignant/output_combiner/part-00000) \
  && echo "Résultat identique avec et sans combiner"

echo; echo "=== Avec 2 reducers"
hadoop jar "$STREAMING_JAR" -D mapreduce.job.reduces=2 -files mapper.py,reducer.py \
  -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
  -input /tp_avance/ca_enseignant/input -output /tp_avance/ca_enseignant/output_2r > /dev/null 2>&1
for f in $(hdfs dfs -ls /tp_avance/ca_enseignant/output_2r | awk '/part-/{print $8}'); do echo "--- $f"; hdfs dfs -cat $f; done

## Sauvegarder les preuves dans l'espace partagé (Drive)

In [ ]:
!hdfs dfs -ls -R /tp_avance > /content/preuves_tp3/arborescence_hdfs.txt
from google.colab import drive
import shutil
drive.mount('/content/drive')
dest = "/content/drive/MyDrive/TP_Hadoop_partage/preuves_tp3"
shutil.copytree("/content/preuves_tp3", dest, dirs_exist_ok=True)
print("Preuves copiées dans", dest, ":", sorted(os.listdir(dest)))

---
# Synthèse des réponses (détail complet dans `docs/TP3_Guide_oral.md`)

| Question | Réponse |
|---|---|
| **15** | `for ligne in sys.stdin` : Hadoop Streaming envoie chaque ligne de l'*input split* sur l'entrée standard du script. |
| **16** | 5 champs ; pays non vide ; quantité entière > 0 ; prix numérique > 0 ; en-tête ignoré. Une ligne rejetée = un compteur. |
| **17** | `pays<TAB>montant` sur stdout. La tabulation sépare la **clé** (utilisée pour le tri et le regroupement) de la **valeur**. |
| **18** | `reporter:counter:Qualite,Pays_manquant,1`, `…Quantite_invalide…`, `…Valeur_negative…` sur **stderr**. |
| **Reducer** | Somme par changement de clé (entrée triée), arrondi `ROUND_HALF_UP` à 0,01, aucun pays codé en dur. |
| **C** | Mapper : 6 paires dans l'ordre du fichier. Final : 3 lignes triées et agrégées (BE 55,00 ; ES 30,00 ; FR 66,00). `sort` regroupe les clés identiques comme le Shuffle/Sort. |
| **D** | Job sur YARN : 2 maps, 1 reduce, `part-00000` + `_SUCCESS`, résultat identique au test local. |
| **19** | Map input records = 10, Map output records = 6, Reduce output records = 3. |
| **20** | En_tete = 1, Lignes_valides = 6, Pays_manquant = 1, Quantite_invalide = 1, Valeur_negative = 1. |
| **21** | 10 lues = 1 en-tête + 6 valides + 3 rejetées ; 6 paires → 3 groupes (3 pays distincts). |
| **22** | `Output directory … already exists`, détecté avant soumission. Correction : `hdfs dfs -rm -r -f` ou nouveau dossier de sortie. |